In [1]:
import pandas as pd
import numpy as np
import random
import torch
import os
import csv
import time
import gc
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from nltk.tokenize import word_tokenize
from itertools import combinations
import random
from gensim.models import LdaModel
import spacy

import sys
sys.path.append('./tools')
from Matave import Matave

In [2]:
K_RANGE = list(range(3, 20)) # chosen to prevent very large numbers of topics in synthetic data generation step
TOP_N = 10

nlp = spacy.load(
    "en_core_web_sm",
    disable=["ner", "parser"]  # speed
)

In [3]:
# Random States
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
domains = {
    'yahoo': 'non-factoid question',
    'banking77': 'banking text',
    'huffPostNews': 'news',
    'clinc150': 'multi-domain intent',
    'atis': 'air travel information system',
    'medicalAbstracts': 'medical abstract (current patient condition)',
    'dementiaAudio': 'dementia cookie theft picture description',
    'syntheticCareHomeNurseNotes': 'nursing home resident',
    'clinicalDialogueSummarizations': 'clinical note',
    'simSUM': 'compact clinical note'
}

In [6]:
# remove punctuation, make lowercase, remove stopwords, punctuation, lemmatize, remove documents with less than 5 tokens
def preprocess_texts(texts, min_words = 5):
    cleaned_texts = []
    for doc in nlp.pipe(texts, batch_size=1000):
        tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and token.lemma_ != "-PRON-"
            and token.is_alpha
        ]
        if len(tokens) >= min_words:
            cleaned_texts.append(" ".join(tokens))
    return cleaned_texts

In [7]:
parent_path = '../getText/datasetsPrep'
all_results = []
for folder in os.listdir(f'{parent_path}'):
    if os.path.isdir(f'{parent_path}/{folder}'):
        for file in os.listdir(f'{parent_path}/{folder}'):
            if file.endswith('.csv'):
                temp_dataset_name = file.replace('.csv', '')
                df = pd.read_csv(f'{parent_path}/{folder}/{file}')
                df = df.dropna().sample(frac=1, random_state=RANDOM_STATE)
                print(f"{file}: {len(df)}")
                texts = df['text'].tolist()
                texts = preprocess_texts(texts)
                print(f"{file}: {len(texts)}")


yahoo.csv: 87362
yahoo.csv: 84346
banking77.csv: 13069
banking77.csv: 3726
huffPostNews.csv: 189815
huffPostNews.csv: 160626
clinc150.csv: 23700
clinc150.csv: 5141
atis.csv: 4978
atis.csv: 3265
medicalAbstracts.csv: 14438
medicalAbstracts.csv: 14438
dementiaAudio.csv: 549
dementiaAudio.csv: 549
syntheticCareHomeNurseNotes.csv: 5783
syntheticCareHomeNurseNotes.csv: 5779
clinicalDialogueSummarizations.csv: 3603
clinicalDialogueSummarizations.csv: 2430
simSUM.csv: 10000
simSUM.csv: 10000
